# Time Series Analysis — Applied on Real Dataset
**Airline Passenger Data pe ARIMA & SARIMA Forecasting**

---
## Table of Contents
1. Dataset Overview — Airline Passengers
2. EDA — Trend, Seasonality, Decomposition
3. Stationarity Testing & Transformation
4. ACF/PACF — Model Order Selection
5. ARIMA Model — Fitting & Forecasting
6. SARIMA Model — Seasonal Forecasting
7. Model Comparison & Evaluation
8. Capstone Challenge + Solutions
---

## Section 1: Dataset Overview

### Keywords:
- **Airline Passengers Dataset:** Jan 1949 to Dec 1960, monthly international air travel passengers
- **Classic benchmark:** Time series analysis mein ye dataset standard example hai
- **Multiplicative Seasonality:** Variance bhi badhti hai with trend → log transform zaroori
- **MAE:** Mean Absolute Error — forecast ki average error
- **RMSE:** Root Mean Square Error — large errors ko zyada penalize karta hai

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Airline Passengers dataset (classic 1949-1960)
# Monthly passengers (thousands) — inline data
passengers = [
    112, 118, 132, 129, 121, 135, 148, 148, 136, 119, 104, 118,  # 1949
    115, 126, 141, 135, 125, 149, 170, 170, 158, 133, 114, 140,  # 1950
    145, 150, 178, 163, 172, 178, 199, 199, 184, 162, 146, 166,  # 1951
    171, 180, 193, 181, 183, 218, 230, 242, 209, 191, 172, 194,  # 1952
    196, 196, 236, 235, 229, 243, 264, 272, 237, 211, 180, 201,  # 1953
    204, 188, 235, 227, 234, 264, 302, 293, 259, 229, 203, 229,  # 1954
    242, 233, 267, 269, 270, 315, 364, 347, 312, 274, 237, 278,  # 1955
    284, 277, 317, 313, 318, 374, 413, 405, 355, 306, 271, 306,  # 1956
    315, 301, 356, 348, 355, 422, 465, 467, 404, 347, 305, 336,  # 1957
    340, 318, 362, 348, 363, 435, 491, 505, 404, 359, 310, 337,  # 1958
    360, 342, 406, 396, 420, 472, 548, 559, 463, 407, 362, 405,  # 1959
    417, 391, 419, 461, 472, 535, 622, 606, 508, 461, 390, 432,  # 1960
]

dates = pd.date_range(start='1949-01', periods=144, freq='MS')
ts = pd.Series(passengers, index=dates, name='Passengers')

print("=== Airline Passengers Dataset ===")
print(f"Period: {ts.index[0].strftime('%b %Y')} to {ts.index[-1].strftime('%b %Y')}")
print(f"Total months: {len(ts)}")
print(f"Min: {ts.min()}, Max: {ts.max()}, Mean: {ts.mean():.0f}")
print(f"\nFirst 12 months (1949):")
print(ts.head(12).to_string())

## Section 2: EDA — Trend, Seasonality, Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

fig, ax = plt.subplots(figsize=(12, 4))
ts.plot(ax=ax, color='steelblue', linewidth=1.5)
ax.set_title('Monthly Airline Passengers (1949-1960)', fontsize=13)
ax.set_ylabel('Passengers (thousands)')
ax.set_xlabel('Year')
plt.tight_layout()
plt.show()

print("Observations:")
print("- Strong upward TREND (more people flying each year)")
print("- Clear SEASONALITY (summer = peak, winter = trough)")
print("- Variance increases with trend → MULTIPLICATIVE model better")

In [ ]:
# Seasonal Decompose — Multiplicative model
decomp = seasonal_decompose(ts, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
components = [
    (ts, 'Original', 'steelblue'),
    (decomp.trend, 'Trend', 'red'),
    (decomp.seasonal, 'Seasonal', 'green'),
    (decomp.resid, 'Residual', 'purple')
]

for ax, (data, title, color) in zip(axes, components):
    data.plot(ax=ax, color=color)
    ax.set_ylabel(title)
    ax.grid(alpha=0.3)

plt.suptitle('Multiplicative Decomposition — Airline Passengers', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Monthly seasonality
monthly_avg = ts.groupby(ts.index.month).mean()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(10, 4))
plt.bar(months, monthly_avg.values, color='steelblue', alpha=0.8)
plt.axhline(monthly_avg.mean(), color='red', linestyle='--', label='Annual Mean')
plt.ylabel('Average Passengers')
plt.title('Average Passengers by Month (Seasonality)')
plt.legend()
plt.show()

peak_month = months[monthly_avg.argmax()]
low_month = months[monthly_avg.argmin()]
print(f"Peak month: {peak_month} ({monthly_avg.max():.0f}k passengers)")
print(f"Lowest month: {low_month} ({monthly_avg.min():.0f}k passengers)")

## Section 3: Stationarity & Transformation

### Keywords:
- **Log Transform:** `np.log(ts)` — multiplicative seasonality ko additive banata hai
- **Differencing:** `ts.diff()` — trend remove karta hai
- **Seasonal Differencing:** `ts.diff(12)` — yearly seasonal pattern remove karta hai
- **ADF Test:** Augmented Dickey-Fuller — p<0.05 means stationary

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    p = result[1]
    verdict = 'STATIONARY ✓' if p < 0.05 else 'NON-STATIONARY ✗'
    print(f"{name:<30} ADF stat: {result[0]:7.3f}  p-value: {p:.4f}  → {verdict}")
    return p < 0.05

print("=== Stationarity Tests ===")
ts_log = np.log(ts)
ts_log_diff1 = ts_log.diff().dropna()
ts_log_diff12 = ts_log.diff(12).dropna()
ts_log_diff1_12 = ts_log.diff(12).diff(1).dropna()

adf_test(ts, 'Original')
adf_test(ts_log, 'Log Transform')
adf_test(ts_log_diff1, 'Log + 1st Diff')
adf_test(ts_log_diff12, 'Log + Seasonal Diff(12)')
adf_test(ts_log_diff1_12, 'Log + 1st + Seasonal Diff')

# Plot transformations
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
ts_log.plot(ax=axes[0], color='steelblue', title='Log Transform')
ts_log_diff1.plot(ax=axes[1], color='red', title='Log + 1st Differencing')
ts_log_diff1_12.plot(ax=axes[2], color='green', title='Log + 1st + Seasonal(12) Differencing')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 4: ACF/PACF — ARIMA Order Selection

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ACF & PACF on stationary series
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

plot_acf(ts_log_diff1_12, lags=30, ax=axes[0], alpha=0.05)
axes[0].set_title('ACF — Stationary Series (Log + 1st + Seasonal Diff)\nUsed to find MA order (q) and seasonal MA (Q)')

plot_pacf(ts_log_diff1_12, lags=30, ax=axes[1], alpha=0.05)
axes[1].set_title('PACF — Stationary Series\nUsed to find AR order (p) and seasonal AR (P)')

plt.tight_layout()
plt.show()

print("=== Reading ACF/PACF for SARIMA ===")
print("Non-seasonal:")
print("  PACF cuts off at lag 1 → p = 1 (AR order)")
print("  ACF cuts off at lag 1 → q = 1 (MA order)")
print("  We used 1 differencing → d = 1")
print("\nSeasonal (at lags 12, 24,...):")
print("  PACF spike at lag 12 → P = 1 (Seasonal AR)")
print("  ACF spike at lag 12 → Q = 1 (Seasonal MA)")
print("  We used 1 seasonal differencing → D = 1, s = 12")
print("\n→ SARIMA(1,1,1)(1,1,1,12)")

## Section 5: ARIMA Model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import numpy as np

# Train-test split (last 24 months for testing)
train = ts_log[:-24]
test = ts_log[-24:]

print(f"Training: {len(train)} months ({train.index[0].strftime('%b %Y')} to {train.index[-1].strftime('%b %Y')})")
print(f"Testing:  {len(test)} months ({test.index[0].strftime('%b %Y')} to {test.index[-1].strftime('%b %Y')})")

# Fit ARIMA(1,1,1) on log-transformed data
arima_model = ARIMA(train, order=(1, 1, 1))
arima_result = arima_model.fit()

print(f"\nARIMA(1,1,1) AIC: {arima_result.aic:.2f}")

# Forecast
arima_forecast_log = arima_result.forecast(steps=24)
arima_forecast = np.exp(arima_forecast_log)
test_actual = np.exp(test)

mae_arima = np.mean(np.abs(test_actual.values - arima_forecast.values))
print(f"ARIMA Test MAE: {mae_arima:.1f} thousand passengers")

# Plot
plt.figure(figsize=(12, 5))
np.exp(ts_log[:-24]).plot(label='Training', color='steelblue')
test_actual.plot(label='Actual Test', color='green')
arima_forecast.plot(label='ARIMA Forecast', color='red', linestyle='--', linewidth=2)
plt.ylabel('Passengers (thousands)')
plt.title('ARIMA(1,1,1) Forecast')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Section 6: SARIMA Model — Seasonal

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# SARIMA(1,1,1)(1,1,1,12) — captures seasonality
sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_result = sarima_model.fit(disp=False)

print(f"SARIMA(1,1,1)(1,1,1,12) AIC: {sarima_result.aic:.2f}")

# Forecast
sarima_forecast_log = sarima_result.forecast(steps=24)
sarima_forecast = np.exp(sarima_forecast_log)

mae_sarima = np.mean(np.abs(test_actual.values - sarima_forecast.values))
print(f"SARIMA Test MAE: {mae_sarima:.1f} thousand passengers")

# Comparison plot
plt.figure(figsize=(13, 5))
np.exp(ts_log[:-24]).plot(label='Training', color='steelblue')
test_actual.plot(label='Actual Test', color='green', linewidth=2)
arima_forecast.plot(label=f'ARIMA MAE={mae_arima:.1f}', color='red', linestyle='--')
sarima_forecast.plot(label=f'SARIMA MAE={mae_sarima:.1f}', color='purple', linestyle='--', linewidth=2)
plt.ylabel('Passengers (thousands)')
plt.title('ARIMA vs SARIMA Forecast Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"\nSARIMA is {(mae_arima-mae_sarima)/mae_arima*100:.1f}% better than ARIMA!")
print("Because SARIMA explicitly models the seasonal pattern (s=12)")

In [ ]:
# Future Forecast — next 12 months beyond dataset
# Refit on all data
sarima_full = SARIMAX(
    ts_log,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

forecast_result = sarima_full.get_forecast(steps=12)
forecast_mean = np.exp(forecast_result.predicted_mean)
conf_int = np.exp(forecast_result.conf_int())

plt.figure(figsize=(13, 5))
np.exp(ts_log[-36:]).plot(label='Historical (last 3 years)', color='steelblue', linewidth=2)
forecast_mean.plot(label='12-Month Forecast', color='red', linewidth=2)
plt.fill_between(conf_int.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                 alpha=0.2, color='red', label='95% Confidence')
plt.ylabel('Passengers (thousands)')
plt.title('SARIMA: 12-Month Future Forecast (Jan–Dec 1961)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("\n=== 1961 Forecast ===")
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
for month, val in zip(months, forecast_mean.values):
    print(f"  1961 {month}: {val:.0f}k passengers")

## Section 7: Aaj Humne Kya Seekha?

### Summary:
1. **Airline Passengers** — Classic real-world time series with trend + seasonality
2. **Log Transform** → multiplicative → additive (makes analysis easier)
3. **Differencing** → non-stationary → stationary (required for ARIMA)
4. **ARIMA(1,1,1)** — basic model, misses seasonal pattern
5. **SARIMA(1,1,1)(1,1,1,12)** — adds seasonal terms, much better!
6. **MAE comparison** — SARIMA clearly wins on seasonal data

### Real-World Uses:
- **Airline/Hospitality:** Demand forecasting, pricing
- **Retail:** Inventory planning for seasonal products
- **Finance:** Stock price forecasting
- **Energy:** Electricity demand forecasting
- **Weather:** Temperature, rainfall patterns

---
## Capstone Challenge

**Problem:** Monthly temperature data pe time series analysis.

**Tasks:**
1. Create monthly temperature data for 5 years (use: `20 + 10*sin(2πt/12) + 0.1t + noise`)
2. Plot the series and identify trend/seasonality
3. Decompose using `seasonal_decompose`
4. Run ADF test — is it stationary?
5. If not, apply differencing to make it stationary
6. Plot ACF and PACF to determine p, d, q
7. Fit SARIMA(1,1,1)(1,1,1,12) on first 48 months
8. Forecast last 12 months and compute MAE

In [ ]:
# CAPSTONE — TRY YOURSELF FIRST!
import numpy as np
import pandas as pd

np.random.seed(42)
t = np.arange(60)
temp = 20 + 10 * np.sin(2 * np.pi * t / 12) + 0.1 * t + np.random.normal(0, 1, 60)
dates_temp = pd.date_range('2019-01', periods=60, freq='MS')
ts_temp = pd.Series(temp.round(1), index=dates_temp, name='Temperature')

print("Temperature series created!")
print(ts_temp.head(12).to_string())
print("Your turn! Complete the 8 tasks...")
# YOUR CODE HERE


---
## Solutions — Capstone

In [ ]:
# === SOLUTION ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Task 1: Create data
np.random.seed(42)
t = np.arange(60)
temp = 20 + 10 * np.sin(2 * np.pi * t / 12) + 0.1 * t + np.random.normal(0, 1, 60)
dates_temp = pd.date_range('2019-01', periods=60, freq='MS')
ts_temp = pd.Series(temp.round(1), index=dates_temp, name='Temperature')

# Task 2: Plot
fig, ax = plt.subplots(figsize=(12, 4))
ts_temp.plot(ax=ax, color='steelblue', linewidth=1.5, marker='o', markersize=4)
ax.set_title('Monthly Temperature (2019-2023)')
ax.set_ylabel('Temperature (°C)')
plt.tight_layout()
plt.show()

# Task 3: Decompose
decomp = seasonal_decompose(ts_temp, model='additive', period=12)
fig, axes = plt.subplots(4, 1, figsize=(12, 8))
for ax, data, title in zip(axes, 
    [ts_temp, decomp.trend, decomp.seasonal, decomp.resid],
    ['Original', 'Trend', 'Seasonal', 'Residual']):
    data.plot(ax=ax)
    ax.set_ylabel(title)
plt.suptitle('Additive Decomposition', fontweight='bold')
plt.tight_layout()
plt.show()

# Task 4: ADF test
def adf_test(series, name=''):
    r = adfuller(series.dropna())
    verdict = 'STATIONARY ✓' if r[1] < 0.05 else 'NON-STATIONARY ✗'
    print(f"{name}: p={r[1]:.4f} → {verdict}")
    return r[1] < 0.05

adf_test(ts_temp, 'Original')

# Task 5: Differencing
ts_diff = ts_temp.diff(1).dropna()
adf_test(ts_diff, 'After 1st diff')

# Task 6: ACF/PACF
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
plot_acf(ts_diff, lags=24, ax=axes[0])
plot_pacf(ts_diff, lags=24, ax=axes[1])
plt.tight_layout()
plt.show()

# Task 7-8: SARIMA forecast
train = ts_temp[:48]
test = ts_temp[48:]

sarima = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12),
                 enforce_stationarity=False, enforce_invertibility=False)
result = sarima.fit(disp=False)

forecast = result.forecast(steps=12)
mae = np.mean(np.abs(test.values - forecast.values))

plt.figure(figsize=(12, 5))
train.plot(label='Training', color='steelblue')
test.plot(label='Actual', color='green', linewidth=2)
forecast.plot(label=f'SARIMA Forecast (MAE={mae:.2f}°C)', color='red', linestyle='--', linewidth=2)
plt.title('Temperature Forecast — SARIMA(1,1,1)(1,1,1,12)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"\nSARIMA MAE: {mae:.2f}°C — excellent for seasonal temperature data!")
print(f"Training period: {len(train)} months, Test period: {len(test)} months")